## BEACON Machine Learning — Log Product Weight Experiment

### Experiment 02: Log-Transformed Product Weight

#### Purpose

This experiment investigates whether logarithmic transformation of
Product Weight improves PCF prediction performance.

The PCF target remains on its original scale. Only the Product Weight
feature is transformed.

The experiment follows the same dataset, train-test split,
preprocessing strategy, regression models and evaluation metrics used
in Experiment 01.

### Experimental comparison

Experiment 01:
- Raw Product Weight
- Raw PCF target

Experiment 02:
- Log-transformed Product Weight
- Raw PCF target

The purpose is therefore to isolate the effect of the Product Weight
transformation.

### Evaluation metrics

- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)
- R²

In [10]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================


import pandas as pd
import numpy as np
from pathlib import Path
DATA_DIR = Path("../data")

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV
)

from sklearn.preprocessing import OneHotEncoder

from category_encoders import TargetEncoder

import warnings

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

In [ ]:
df = pd.read_csv(
    DATA_DIR / "PublicTablesForCarbonCatalogueDataDescriptor_v30Oct2021(Product Level Data).csv",
    encoding="latin1"
)


In [27]:
df.columns


Index(['*PCF-ID', 'Year of reporting', '*Stage-level CO2e available',
       'Product name (and functional unit)', 'Product detail', 'Company',
       'Country (where company is incorporated)',
       'Company's GICS Industry Group', 'Company's GICS Industry',
       '*Company's sector', 'Product weight (kg)',
       '*Source for product weight',
       'Product's carbon footprint (PCF, kg CO2e)', '*Carbon intensity',
       'Protocol used for PCF', 'Relative change in PCF vs previous',
       'Company-reported reason for change', '*Change reason category',
       '*%Upstream estimated from %Operations',
       '*Upstream CO2e (fraction of total PCF)',
       '*Operations CO2e (fraction of total PCF)',
       '*Downstream CO2e (fraction of total PCF)',
       '*Transport CO2e (fraction of total PCF)',
       '*EndOfLife CO2e (fraction of total PCF)',
       '*Adjustments to raw data (if any)'],
      dtype='str')

In [28]:
# ============================================================
# 3. CREATE ML DATASET
# ============================================================

product_df = df.rename(columns={
    "Year of reporting": "Year",

    "Company": "Company",
    "Country (where company is incorporated)": "Country",
    "Company's GICS Industry": "Industry",

    "Product weight (kg)": "Product_Weight",

    "Protocol used for PCF": "PCF_Protocol",

   "*Stage-level CO2e available":"Stage_Level_CO2e_Available",

    "Product's carbon footprint (PCF, kg CO2e)": "PCF"
})

print("ML dataset shape:", product_df.shape)

display(product_df.head())

df_ml = product_df

ML dataset shape: (866, 25)


,*PCF-ID,Year,Stage_Level_CO2e_Available,Product name (and functional unit),Product detail,Company,Country,Company's GICS Industry Group,Industry,*Company's sector,...,Relative change in PCF vs previous,Company-reported reason for change,*Change reason category,*%Upstream estimated from %Operations,*Upstream CO2e (fraction of total PCF),*Operations CO2e (fraction of total PCF),*Downstream CO2e (fraction of total PCF),*Transport CO2e (fraction of total PCF),*EndOfLife CO2e (fraction of total PCF),*Adjustments to raw data (if any)
0,10056-1-2014,2014,Yes,Frosted Flakes(R) Cereal,"Frosted Flakes(R), 23 oz., Produced in Lancast...",Kellogg Company,USA,"Food, Beverage & Tobacco",Food Products,Food & Beverage,...,(not reported by company),N/a,N/a (no %change reported),No,57.50%,30.00%,12.50%,4.50%,(included in downstream but not reported separ...,Divided stage and total emissions by 1000 (bas...
1,10056-1-2015,2015,Yes,"Frosted Flakes, 23 oz, produced in Lancaster, ...",Cereal,Kellogg Company,USA,Food & Beverage Processing,Not used for 2015 reporting,Food & Beverage,...,(not reported by company),N/a,N/a (no %change reported),No,57.50%,30.00%,12.50%,4.50%,(included in downstream but not reported separ...,Divided stage and total emissions by 1000 (bas...
2,10222-1-2013,2013,Yes,Office Chair,Field not included in 2013 data,KNOLL INC,USA,Capital Goods,Building Products,Comm. equipm. & capital goods,...,(not reported by company),N/a,N/a (no previous data available),Yes,80.63%,17.36%,2.01%,(included in up/downstream but not reported se...,0.00%,"Changed %change to zero, according to field ""c..."
3,10261-1-2017,2017,Yes,Multifunction Printers,bizhub C458,"Konica Minolta, Inc.",Japan,Technology Hardware & Equipment,"Electronic Equipment, Instruments & Components","Computer, IT & telecom",...,(not reported by company),N/a,N/a (no previous data available),No,30.65%,5.51%,63.84%,1.01%,2.76%,NaN
4,10261-2-2017,2017,Yes,Multifunction Printers,bizhub C558,"Konica Minolta, Inc.",Japan,Technology Hardware & Equipment,"Electronic Equipment, Instruments & Components","Computer, IT & telecom",...,(not reported by company),N/a,N/a (no previous data available),No,25.08%,4.51%,70.41%,0.83%,2.26%,NaN


In [30]:
# ============================================================
# 4. CLEAN TARGET
# ============================================================

df_ml["PCF"] = pd.to_numeric(
    df_ml["PCF"],
    errors="coerce"
)

df_ml = df_ml[
    df_ml["PCF"].notna()
].copy()

df_ml = df_ml[
    df_ml["PCF"] >= 0
].copy()

print("Usable rows:", len(df_ml))

print("\nPCF summary:")
display(df_ml["PCF"].describe())

print(
    "\nPCF skewness:",
    df_ml["PCF"].skew()
)

Usable rows: 866

PCF summary:


count    8.660000e+02
mean     1.581525e+04
std      1.813733e+05
min      4.000000e-04
25%      7.000000e+00
50%      1.111000e+02
75%      1.600000e+03
max      3.718044e+06
Name: PCF, dtype: float64


PCF skewness: 17.63189911896261


In [31]:
# ============================================================
# 5. DEFINE FEATURES AND TARGET
# ============================================================

FEATURES = [
    "Year",
    "Company",
    "Stage_Level_CO2e_Available",
    "Country",
    "Industry",
    "Product_Weight",
    "PCF_Protocol"
]

TARGET = "PCF"

X = df_ml[FEATURES].copy()

y = df_ml[TARGET].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (866, 7)
y shape: (866,)


In [ ]:
# ============================================================
# 6. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print("Training rows:", len(X_train))
print("Testing rows :", len(X_test))

Training rows: 692
Testing rows : 174


In [ ]:
# ============================================================
# 7. CREATE SAFE COPIES
# ============================================================

X_train = X_train.copy()
X_test = X_test.copy()

y_train = y_train.copy()
y_test = y_test.copy()

In [ ]:
# ============================================================
# 8. COUNTRY → REGION
# ============================================================

country_to_region = {

    # North America
    "USA": "North America",
    "Canada": "North America",

    # Europe
    "Germany": "Europe",
    "Netherlands": "Europe",
    "United Kingdom": "Europe",
    "Switzerland": "Europe",
    "Sweden": "Europe",
    "Finland": "Europe",
    "Italy": "Europe",
    "France": "Europe",
    "Spain": "Europe",
    "Belgium": "Europe",
    "Ireland": "Europe",
    "Luxembourg": "Europe",
    "Lithuania": "Europe",
    "Greece": "Europe",

    # East Asia
    "Japan": "East Asia",
    "Taiwan": "East Asia",
    "South Korea": "East Asia",
    "China": "East Asia",

    # South Asia
    "India": "South Asia",

    # Southeast Asia
    "Malaysia": "Southeast Asia",
    "Indonesia": "Southeast Asia",

    # South America
    "Brazil": "South America",
    "Chile": "South America",
    "Colombia": "South America",

    # Africa
    "South Africa": "Africa",

    # Oceania
    "Australia": "Oceania"
}

X_train["Region"] = X_train[
    "Country"
].map(country_to_region)

X_test["Region"] = X_test[
    "Country"
].map(country_to_region)

X_train["Region"] = (
    X_train["Region"].fillna("Other")
)

X_test["Region"] = (
    X_test["Region"].fillna("Other")
)

X_train.drop(
    columns="Country",
    inplace=True
)

X_test.drop(
    columns="Country",
    inplace=True
)

In [ ]:
# ============================================================
# 9. RARE INDUSTRY → OTHER
#    FIT USING TRAINING DATA ONLY
# ============================================================

industry_counts = X_train["Industry"].value_counts()

rare_industries = industry_counts[industry_counts < 10].index
X_train["Industry"] = X_train["Industry"].replace(rare_industries, "Other")

X_test["Industry"] = X_test[
    "Industry"
].replace(
    rare_industries,
    "Other"
)

print(
    "Rare industries grouped:",
    len(rare_industries)
)

Rare industries grouped: 17


In [ ]:
# ============================================================
# 10. PCF PROTOCOL → TOP 5 + OTHER
# ============================================================

top_protocols = [
    "ISO",
    "Not reported",
    "GHGP",
    "PAS2050",
    "TRACI 2.1"
]

X_train["PCF_Protocol"] = X_train[
    "PCF_Protocol"
].where(
    X_train["PCF_Protocol"].isin(
        top_protocols
    ),
    "Other"
)

X_test["PCF_Protocol"] = X_test[
    "PCF_Protocol"
].where(
    X_test["PCF_Protocol"].isin(
        top_protocols
    ),
    "Other"
)

In [ ]:
# ============================================================
# 11. STAGE-LEVEL CO2e → BINARY
# ============================================================

binary_map = {
    "No": 0,
    "Yes": 1
}

X_train[
    "Stage_Level_CO2e_Available"
] = (
    X_train[
        "Stage_Level_CO2e_Available"
    ]
    .map(binary_map)
    .fillna(0)
)

X_test[
    "Stage_Level_CO2e_Available"
] = (
    X_test[
        "Stage_Level_CO2e_Available"
    ]
    .map(binary_map)
    .fillna(0)
)

In [ ]:
# ============================================================
# 11. STAGE-LEVEL CO2e → BINARY
# ============================================================

binary_map = {
    "No": 0,
    "Yes": 1
}

X_train[
    "Stage_Level_CO2e_Available"
] = (
    X_train[
        "Stage_Level_CO2e_Available"
    ]
    .map(binary_map)
    .fillna(0)
)

X_test[
    "Stage_Level_CO2e_Available"
] = (
    X_test[
        "Stage_Level_CO2e_Available"
    ]
    .map(binary_map)
    .fillna(0)
)

In [ ]:
# ============================================================
# 12. PRODUCT WEIGHT — WINSORISATION
#
# Purpose:
# Control extreme Product Weight values using the training
# distribution before applying the logarithmic transformation.
# ============================================================

Q1 = X_train[
    "Product_Weight"
].quantile(0.25)

Q3 = X_train[
    "Product_Weight"
].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print("Winsorisation bounds")
print("Lower:", lower)
print("Upper:", upper)

X_train[
    "Product_Weight"
] = X_train[
    "Product_Weight"
].clip(
    lower=lower,
    upper=upper
)

X_test[
    "Product_Weight"
] = X_test[
    "Product_Weight"
].clip(
    lower=lower,
    upper=upper
)

Winsorisation bounds
Lower: -1497.5
Upper: 2498.5


In [ ]:
# ============================================================
# 13. LOG-TRANSFORM PRODUCT WEIGHT
#
# log1p(x) = log(1 + x)
#
# This is used because Product Weight is highly right-skewed.
# ============================================================

X_train["Log_Product_Weight"] = np.log1p(
    X_train["Product_Weight"]
)

X_test["Log_Product_Weight"] = np.log1p(
    X_test["Product_Weight"]
)

print("Original Product Weight skewness:")
print(
    X_train["Product_Weight"].skew()
)

print("\nLog Product Weight skewness:")
print(
    X_train["Log_Product_Weight"].skew()
)

Original Product Weight skewness:
1.6714789949895565

Log Product Weight skewness:
0.37597829711470965


In [ ]:
# ============================================================
# 14. REPLACE RAW PRODUCT WEIGHT
#
# The raw Product_Weight feature is removed so that this
# experiment isolates the effect of the logarithmic
# representation.
# ============================================================

X_train.drop(
    columns="Product_Weight",
    inplace=True
)

X_test.drop(
    columns="Product_Weight",
    inplace=True
)

print(
    "Log_Product_Weight present:",
    "Log_Product_Weight" in X_train.columns
)

print(
    "Raw Product_Weight present:",
    "Product_Weight" in X_train.columns
)

Log_Product_Weight present: True
Raw Product_Weight present: False


In [ ]:
# ============================================================
# 15. COMPANY → TARGET ENCODING
#    FIT USING TRAINING DATA ONLY
# ============================================================

te = TargetEncoder(
    cols=["Company"],
    min_samples_leaf=20,
    smoothing=10
)

X_train = te.fit_transform(
    X_train,
    y_train
)

X_test = te.transform(
    X_test
)

In [ ]:
# ============================================================
# 16. ONE-HOT ENCODING
# ============================================================

categorical_features = [
    "Industry",
    "PCF_Protocol",
    "Region"
]

encoder = OneHotEncoder(
    drop="first",
    handle_unknown="ignore",
    sparse_output=False
)

encoded_train = encoder.fit_transform(
    X_train[
        categorical_features
    ]
)

encoded_test = encoder.transform(
    X_test[
        categorical_features
    ]
)

encoded_train_df = pd.DataFrame(
    encoded_train,
    columns=encoder.get_feature_names_out(
        categorical_features
    ),
    index=X_train.index
)

encoded_test_df = pd.DataFrame(
    encoded_test,
    columns=encoder.get_feature_names_out(
        categorical_features
    ),
    index=X_test.index
)

X_train = pd.concat(
    [
        X_train.drop(
            columns=categorical_features
        ),
        encoded_train_df
    ],
    axis=1
)

X_test = pd.concat(
    [
        X_test.drop(
            columns=categorical_features
        ),
        encoded_test_df
    ],
    axis=1
)

In [ ]:
# ============================================================
# 17. FINAL FEATURE CHECK
# ============================================================

print(
    "Final X_train shape:",
    X_train.shape
)

print(
    "Final X_test shape:",
    X_test.shape
)

print(
    "\nLog Product Weight present:",
    "Log_Product_Weight" in X_train.columns
)

print(
    "Raw Product Weight present:",
    "Product_Weight" in X_train.columns
)

print(
    "Log target used:",
    False
)

print(
    "\nAll features numerical:",
    X_train.select_dtypes(
        exclude="number"
    ).empty
)

print("\nFinal columns:")
print(
    X_train.columns.tolist()
)

Final X_train shape: (692, 34)
Final X_test shape: (174, 34)

Log Product Weight present: True
Raw Product Weight present: False
Log target used: False

All features numerical: True

Final columns:
['Year', 'Company', 'Stage_Level_CO2e_Available', 'Log_Product_Weight', 'Industry_Automobiles', 'Industry_Beverages', 'Industry_Chemicals', 'Industry_Commercial Services & Supplies', 'Industry_Computers & Peripherals', 'Industry_Containers & Packaging', 'Industry_Electrical Equipment', 'Industry_Electronic Equipment, Instruments & Components', 'Industry_Food & Staples Retailing', 'Industry_Food Products', 'Industry_Household Durables', 'Industry_Metals & Mining', 'Industry_Not used for 2015 reporting', 'Industry_Office Electronics', 'Industry_Other', 'Industry_Paper & Forest Products', 'Industry_Software', 'Industry_Textiles, Apparel & Luxury Goods', 'PCF_Protocol_ISO', 'PCF_Protocol_Not reported', 'PCF_Protocol_Other', 'PCF_Protocol_PAS2050', 'PCF_Protocol_TRACI 2.1', 'Region_East Asia', 'R

In [ ]:
# ============================================================
# 18. DEFINE BASELINE ML MODELS
# ============================================================

models = {

    "Linear Regression":
        LinearRegression(),

    "ElasticNet":
        ElasticNet(
            random_state=RANDOM_STATE
        ),

    "Bayesian Ridge":
        BayesianRidge(),

    "Random Forest":
        RandomForestRegressor(
            random_state=RANDOM_STATE
        ),

    "Extra Trees":
        ExtraTreesRegressor(
            random_state=RANDOM_STATE
        ),

    "HistGradientBoosting":
        HistGradientBoostingRegressor(
            random_state=RANDOM_STATE
        ),

    "XGBoost":
        XGBRegressor(
            random_state=RANDOM_STATE
        )
}

In [ ]:
# ============================================================
# 19. BASELINE MODEL EVALUATION
# ============================================================

baseline_results = []

for name, model in models.items():

    print(
        f"Training: {name}"
    )

    model.fit(
        X_train,
        y_train
    )

    y_pred = model.predict(
        X_test
    )

    mae = mean_absolute_error(
        y_test,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            y_pred
        )
    )

    r2 = r2_score(
        y_test,
        y_pred
    )

    baseline_results.append({

        "Model": name,

        "MAE": mae,

        "RMSE": rmse,

        "R²": r2
    })

Training: Linear Regression
Training: ElasticNet
Training: Bayesian Ridge
Training: Random Forest
Training: Extra Trees
Training: HistGradientBoosting
Training: XGBoost


In [ ]:
# ============================================================
# 20. BASELINE RESULTS
# ============================================================

baseline_results_df = pd.DataFrame(
    baseline_results
)

baseline_results_df = (
    baseline_results_df
    .sort_values(
        "R²",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    baseline_results_df.style.format({
        "MAE": "{:,.2f}",
        "RMSE": "{:,.2f}",
        "R²": "{:.4f}"
    })
)

,Model,MAE,RMSE,R²
0,Random Forest,"2,633.02","8,761.51",0.4941
1,XGBoost,"2,657.41","9,153.94",0.4477
2,Extra Trees,"2,813.94","9,569.85",0.3964
3,Linear Regression,"18,215.17","23,615.91",-2.6757
4,ElasticNet,"21,194.36","29,001.03",-4.5431
5,Bayesian Ridge,"21,751.64","30,182.26",-5.0038
6,HistGradientBoosting,"11,237.46","34,806.17",-6.9843


In [ ]:
# ============================================================
# 21. SELECT MODELS FOR HYPERPARAMETER TUNING
# ============================================================

tuning_models = {

    "Random Forest":
        RandomForestRegressor(
            random_state=RANDOM_STATE
        ),

    "Extra Trees":
        ExtraTreesRegressor(
            random_state=RANDOM_STATE
        ),

    "XGBoost":
        XGBRegressor(
            random_state=RANDOM_STATE
        )
}

print(
    "Models selected for tuning:"
)

for name in tuning_models:
    print("-", name)

Models selected for tuning:
- Random Forest
- Extra Trees
- XGBoost


In [ ]:
# ============================================================
# 22. RANDOM FOREST — GRID SEARCH
# ============================================================

rf_param_grid = {

    "n_estimators": [
        100,
        200,
        300
    ],

    "max_depth": [
        None,
        10,
        20
    ],

    "min_samples_split": [
        2,
        5
    ],

    "min_samples_leaf": [
        1,
        2
    ]
}

rf_grid = GridSearchCV(
    estimator=tuning_models[
        "Random Forest"
    ],

    param_grid=rf_param_grid,

    scoring="neg_root_mean_squared_error",

    cv=5,

    n_jobs=-1,

    verbose=1
)

rf_grid.fit(
    X_train,
    y_train
)

print(
    "\nBest Random Forest parameters:"
)

print(
    rf_grid.best_params_
)

Fitting 5 folds for each of 36 candidates, totalling 180 fits

Best Random Forest parameters:
{'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100}


In [ ]:
# ============================================================
# 23. EVALUATE TUNED RANDOM FOREST
# ============================================================

best_rf = rf_grid.best_estimator_

rf_pred = best_rf.predict(
    X_test
)

rf_mae = mean_absolute_error(
    y_test,
    rf_pred
)

rf_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        rf_pred
    )
)

rf_r2 = r2_score(
    y_test,
    rf_pred
)

print("Random Forest")
print("MAE :", rf_mae)
print("RMSE:", rf_rmse)
print("R²  :", rf_r2)

Random Forest
MAE : 2677.047494669272
RMSE: 8864.627835530839
R²  : 0.4820990991119527


In [ ]:
# ============================================================
# 24. EXTRA TREES — GRID SEARCH
# ============================================================

et_param_grid = {

    "n_estimators": [
        100,
        200,
        300
    ],

    "max_depth": [
        None,
        10,
        20
    ],

    "min_samples_split": [
        2,
        5
    ],

    "min_samples_leaf": [
        1,
        2
    ]
}

et_grid = GridSearchCV(
    estimator=tuning_models[
        "Extra Trees"
    ],

    param_grid=et_param_grid,

    scoring="neg_root_mean_squared_error",

    cv=5,

    n_jobs=-1,

    verbose=1
)

et_grid.fit(
    X_train,
    y_train
)

print(
    "\nBest Extra Trees parameters:"
)

print(
    et_grid.best_params_
)

Fitting 5 folds for each of 36 candidates, totalling 180 fits

Best Extra Trees parameters:
{'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}


In [ ]:
# ============================================================
# 25. EVALUATE TUNED EXTRA TREES
# ============================================================

best_et = et_grid.best_estimator_

et_pred = best_et.predict(
    X_test
)

et_mae = mean_absolute_error(
    y_test,
    et_pred
)

et_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        et_pred
    )
)

et_r2 = r2_score(
    y_test,
    et_pred
)

print("Extra Trees")
print("MAE :", et_mae)
print("RMSE:", et_rmse)
print("R²  :", et_r2)

Extra Trees
MAE : 2598.4497328603616
RMSE: 8801.647755888735
R²  : 0.48943196696324376


In [ ]:
# ============================================================
# 26. XGBOOST — GRID SEARCH
# ============================================================

xgb_param_grid = {

    "n_estimators": [
        100,
        200,
        300
    ],

    "max_depth": [
        3,
        5,
        7
    ],

    "learning_rate": [
        0.01,
        0.05,
        0.1
    ],

    "subsample": [
        0.8,
        1.0
    ],

    "colsample_bytree": [
        0.8,
        1.0
    ]
}

xgb_grid = GridSearchCV(
    estimator=tuning_models[
        "XGBoost"
    ],

    param_grid=xgb_param_grid,

    scoring="neg_root_mean_squared_error",

    cv=5,

    n_jobs=-1,

    verbose=1
)

xgb_grid.fit(
    X_train,
    y_train
)

print(
    "\nBest XGBoost parameters:"
)

print(
    xgb_grid.best_params_
)

Fitting 5 folds for each of 108 candidates, totalling 540 fits

Best XGBoost parameters:
{'colsample_bytree': 1.0, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.8}


In [ ]:
# ============================================================
# 27. EVALUATE TUNED XGBOOST
# ============================================================

best_xgb = xgb_grid.best_estimator_

xgb_pred = best_xgb.predict(
    X_test
)

xgb_mae = mean_absolute_error(
    y_test,
    xgb_pred
)

xgb_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        xgb_pred
    )
)

xgb_r2 = r2_score(
    y_test,
    xgb_pred
)

print("XGBoost")
print("MAE :", xgb_mae)
print("RMSE:", xgb_rmse)
print("R²  :", xgb_r2)

XGBoost
MAE : 4664.92129165634
RMSE: 9102.309232135105
R²  : 0.4539545115231841


In [ ]:
# ============================================================
# 28. TUNED MODEL COMPARISON
# ============================================================

tuned_results_df = pd.DataFrame([

    {
        "Model": "Random Forest",
        "MAE": rf_mae,
        "RMSE": rf_rmse,
        "R²": rf_r2
    },

    {
        "Model": "Extra Trees",
        "MAE": et_mae,
        "RMSE": et_rmse,
        "R²": et_r2
    },

    {
        "Model": "XGBoost",
        "MAE": xgb_mae,
        "RMSE": xgb_rmse,
        "R²": xgb_r2
    }

])

tuned_results_df = (
    tuned_results_df
    .sort_values(
        "R²",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    tuned_results_df.style.format({
        "MAE": "{:,.2f}",
        "RMSE": "{:,.2f}",
        "R²": "{:.4f}"
    })
)

,Model,MAE,RMSE,R²
0,Extra Trees,"2,598.45","8,801.65",0.4894
1,Random Forest,"2,677.05","8,864.63",0.4821
2,XGBoost,"4,664.92","9,102.31",0.4540


In [ ]:
# ============================================================
# 29. BASELINE VS TUNED
# ============================================================

baseline_selected = (
    baseline_results_df[
        baseline_results_df["Model"].isin([
            "Random Forest",
            "Extra Trees",
            "XGBoost"
        ])
    ][
        ["Model", "MAE", "RMSE", "R²"]
    ]
    .copy()
)

baseline_selected["Stage"] = "Baseline"

tuned_comparison = tuned_results_df.copy()

tuned_comparison["Stage"] = "Tuned"

baseline_vs_tuned = pd.concat(
    [
        baseline_selected,
        tuned_comparison
    ],
    ignore_index=True
)

display(
    baseline_vs_tuned.sort_values(
        ["Model", "Stage"]
    )
)

,Model,MAE,RMSE,R²,Stage
2,Extra Trees,2813.935672,9569.849636,0.396419,Baseline
3,Extra Trees,2598.449733,8801.647756,0.489432,Tuned
0,Random Forest,2633.022354,8761.505154,0.494079,Baseline
4,Random Forest,2677.047495,8864.627836,0.482099,Tuned
1,XGBoost,2657.414002,9153.936535,0.447743,Baseline
5,XGBoost,4664.921292,9102.309232,0.453955,Tuned


In [ ]:
# ============================================================
# 30. SAVE BASELINE RESULTS
# ============================================================

baseline_results_df.to_csv(
    "baseline_log_product_weight_results.csv",
    index=False
)

print(
    "Saved baseline results."
)

Saved baseline results.


In [ ]:
# ============================================================
# 32. SAVE BEST HYPERPARAMETERS
# ============================================================

best_parameters_df = pd.DataFrame({

    "Model": [
        "Random Forest",
        "Extra Trees",
        "XGBoost"
    ],

    "Best Parameters": [
        rf_grid.best_params_,
        et_grid.best_params_,
        xgb_grid.best_params_
    ]
})

display(
    best_parameters_df
)

best_parameters_df.to_csv(
    "tuned_log_product_weight_best_parameters.csv",
    index=False
)

,Model,Best Parameters
0,Random Forest,"{'max_depth': 10, 'min_samples_leaf': 1, 'min_..."
1,Extra Trees,"{'max_depth': None, 'min_samples_leaf': 2, 'mi..."
2,XGBoost,"{'colsample_bytree': 1.0, 'learning_rate': 0.0..."


In [ ]:
# ============================================================
# 33. EXPERIMENT SUMMARY
# ============================================================

print("=" * 80)
print("EXPERIMENT 02 — LOG PRODUCT WEIGHT")
print("=" * 80)

print(
    "\nDataset rows:",
    len(df_ml)
)

print(
    "Training rows:",
    len(X_train)
)

print(
    "Testing rows:",
    len(X_test)
)

print(
    "\nTarget transformation: None"
)

print(
    "Product Weight transformation: log1p"
)

best_baseline = baseline_results_df.iloc[0]

print(
    "\nBest baseline model by R²:"
)

print(
    best_baseline["Model"],
    "→ R² =",
    round(
        best_baseline["R²"],
        4
    )
)

best_tuned = tuned_results_df.iloc[0]

print(
    "\nBest tuned model by R²:"
)

print(
    best_tuned["Model"],
    "→ R² =",
    round(
        best_tuned["R²"],
        4
    )
)

print("\n" + "=" * 80)

EXPERIMENT 02 — LOG PRODUCT WEIGHT

Dataset rows: 866
Training rows: 692
Testing rows: 174

Target transformation: None
Product Weight transformation: log1p

Best baseline model by R²:
Random Forest → R² = 0.4941

Best tuned model by R²:
Extra Trees → R² = 0.4894



# ============================================================
# 33. EXPERIMENT SUMMARY
# ============================================================

print("=" * 80)
print("EXPERIMENT 02 — LOG PRODUCT WEIGHT")
print("=" * 80)

print(
    "\nDataset rows:",
    len(df_ml)
)

print(
    "Training rows:",
    len(X_train)
)

print(
    "Testing rows:",
    len(X_test)
)

print(
    "\nTarget transformation: None"
)

print(
    "Product Weight transformation: log1p"
)

best_baseline = baseline_results_df.iloc[0]

print(
    "\nBest baseline model by R²:"
)

print(
    best_baseline["Model"],
    "→ R² =",
    round(
        best_baseline["R²"],
        4
    )
)

best_tuned = tuned_results_df.iloc[0]

print(
    "\nBest tuned model by R²:"
)

print(
    best_tuned["Model"],
    "→ R² =",
    round(
        best_tuned["R²"],
        4
    )
)

print("\n" + "=" * 80)

## Experiment 02 — Result Interpretation

The log-transformed Product Weight experiment did not improve the
baseline Random Forest performance relative to Experiment 01. The
baseline Random Forest achieved an R² of 0.4941 with log-transformed
Product Weight, compared with 0.5195 when the original Product Weight
representation was used.

Although the log transformation reduced the influence of the highly
skewed Product Weight distribution, the transformation did not translate
into improved predictive performance for the Random Forest model.

Extra Trees showed a notable improvement following hyperparameter
optimisation, increasing R² from 0.3964 to 0.4894 while also reducing
MAE and RMSE. However, the tuned Extra Trees model remained slightly
below the baseline Random Forest performance.

For XGBoost, hyperparameter optimisation produced only a small increase
in R² (0.4477 to 0.4540), while MAE increased substantially. Therefore,
the effect of tuning was not consistently beneficial across all
evaluation metrics.

Overall, Experiment 02 does not provide evidence that replacing the
original Product Weight with its logarithmic representation improves
PCF prediction performance. The results are retained for comparison
with the raw-feature baseline and the subsequent log-target experiment.